# 01_ingest_epss
---
## EPSS Bronze : EPSS Daily CSV를 수집·검증하고 Bronze Raw와 Metadata를 저장하는 모듈.

- 로컬 검증 후 fabric 검증 했던 노트북. 최종 파일명은 01_ingest_epss 



Flow:

download_epss → validate_epss → build_metadata → save_bronze_local


## [1] HTTP 호출하기

### (1) ingest_api.py 불러오기

In [1]:

%run -b ingest_api.py

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 3, Finished, Available, Finished, True)

In [2]:
print(request_with_retry)

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 4, Finished, Available, Finished, False)

<function request_with_retry at 0x737e482c2700>


### (2) bronze 파일 경로 준비

In [3]:
''' 첫 실행 이후 실행하지 않음. 
notebookutils.fs.mkdirs("Files/bronze/01_epss")

print("Bronze EPSS 경로 준비 완료")
'''
# -----------
# 1일1회 스케줄러 테스트용
# -----------
# notebookutils.fs.mkdirs("Files/test_shcedule/bronze_test/01_epss_test_shcedule")

# print("Bronze EPSS test 경로 준비 완료")

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 5, Finished, Available, Finished, False)

' 첫 실행 이후 실행하지 않음. \nnotebookutils.fs.mkdirs("Files/bronze/01_epss")\n\nprint("Bronze EPSS 경로 준비 완료")\n'

### (3) 라이브러리 설치하기

In [4]:
import gzip
import hashlib
import io
import json
from datetime import datetime, timezone, date, timedelta
from pathlib import Path

import pandas as pd

# from builtin.ingest_api import request_with_retry

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 6, Finished, Available, Finished, False)

## [2] EPSS 적재 함수 정의 및 실행

### (1) 함수 정의

In [5]:
# ===============================================
# download_epss()
# ================================================

EPSS_CURRENT_URL = "https://epss.empiricalsecurity.com/epss_scores-current.csv.gz"
EPSS_HISTORICAL_URL = "https://epss.empiricalsecurity.com/epss_scores-{date}.csv.gz"



def download_epss(snapshot_date=None, timeout=30):
    '''EPSS를 받아서 원본 bytes와 수집 실행 정보를 반환'''
    if snapshot_date:
        url = EPSS_HISTORICAL_URL.format(date=snapshot_date)
    else:
        url = EPSS_CURRENT_URL

    started_at = datetime.now(timezone.utc)

    response = request_with_retry(
        "GET",
        url,
        timeout=timeout
    )

    completed_at = datetime.now(timezone.utc)

    return {
        "content": response.content,
        "status_code": response.status_code,
        "final_url": response.url,
        "started_at": started_at,
        "completed_at": completed_at,
        "requested_date": snapshot_date
    }

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 7, Finished, Available, Finished, False)

In [6]:
# ================================================
# validate_epss()
# ================================================

def validate_epss(content):
    '''EPSS Raw를 검증하고 snapshot 기준 정보를 반환'''
    # 1. gzip 해제
    try:
        with gzip.open(
            io.BytesIO(content),
            "rt",
            encoding="utf-8"
        ) as f:
            text = f.read()
    except Exception as e:
        raise ValueError(f"EPSS gzip validation failed: {e}")

    # 2. CSV parsing
    try:
        df = pd.read_csv(
            io.StringIO(text),
            comment="#"
        )
    except Exception as e:
        raise ValueError(f"EPSS CSV parsing failed: {e}")

    # 3. 필수 컬럼 검증
    required_columns = {"cve", "epss", "percentile"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"EPSS required columns missing: {missing_columns}"
        )

    # 4. 0 Row 검증
    if len(df) == 0:
        raise ValueError("EPSS data contains 0 rows")

    # 5. score_date 추출
    score_date = None

    for line in text.splitlines()[:5]:
        if "score_date:" in line:
            score_date = (
                line
                .split("score_date:", 1)[1]
                .split(",", 1)[0]
                .strip()
            )
            break

    if score_date is None:
        raise ValueError("EPSS score_date not found")

    # 날짜 부분만 snapshot_date로 사용
    snapshot_date = score_date.split("T", 1)[0]

    return {
        "df": df,
        "record_count": len(df),
        "score_date": score_date,
        "snapshot_date": snapshot_date
    }

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 8, Finished, Available, Finished, False)

In [7]:
# ================================================
# build_metadata()
# ================================================

def build_metadata(download_result, validation_result):
    '''다운로드 결과와 검증 결과를 받아서, Task 1 계약에 맞는 _meta.json용 딕셔너리를 생성'''
    content = download_result["content"]

    # Raw gzip 파일 자체의 SHA-256
    checksum = hashlib.sha256(content).hexdigest()

    started_at = download_result["started_at"]
    completed_at = download_result["completed_at"]

    snapshot_date = validation_result["snapshot_date"]
    record_count = validation_result["record_count"]

    batch_id = f"epss_{started_at.strftime('%Y%m%dT%H%M%SZ')}"

    file_name = (
        f"epss_{snapshot_date}_"
        f"{started_at.strftime('%Y%m%dT%H%M%SZ')}.csv.gz"
    )

    metadata = {
        "source": "epss",
        "source_url": download_result["final_url"],
        "batch_id": batch_id,
        "fetched_at": completed_at.isoformat(),
        "snapshot_date": snapshot_date,
        "file_name": file_name,
        "format": "csv.gz",
        "record_count": record_count,
        "checksum": f"sha256:{checksum}",
        "schema_version": "bronze-v1"
    }

    return metadata

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 9, Finished, Available, Finished, False)

In [8]:
# ================================================
# save_bronze_fabric()
# ================================================

def save_bronze_fabric(content, metadata, bronze_dir="Files/bronze/01_epss"):
    import json
    import notebookutils

    notebookutils.fs.mkdirs(bronze_dir)

    file_name = metadata["file_name"]
    raw_path = f"{bronze_dir}/{file_name}"
    meta_path = f"{bronze_dir}/{file_name.replace('.csv.gz', '_meta.json')}"

    # binary Raw는 임시 로컬 파일로 저장 후 Lakehouse로 복사(binary byte 형태를 문자열 utf-8로 안전하게 쓰기 위함. )
    local_raw_path = f"/tmp/{file_name}"

    with open(local_raw_path, "wb") as f:
        f.write(content)

    notebookutils.fs.cp(
        f"file:{local_raw_path}",
        raw_path
    )

    # Metadata는 JSON 문자열로 직접 저장
    notebookutils.fs.put(
        meta_path,
        json.dumps(metadata, indent=2, ensure_ascii=False),
        overwrite=False
    )

    return {
        "raw_path": raw_path,
        "meta_path": meta_path
    }

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 10, Finished, Available, Finished, False)

In [9]:
# ====================================
# backfill_epss()
# ====================================
def backfill_epss(
    start_date="2025-03-21",
    end_date=None,
    bronze_dir="Files/bronze/01_epss"
):
    '''2025-03-21부터 특정일까지 데이터 수집하는 함수'''
    if end_date is None:
        end_date = datetime.now(timezone.utc).date().isoformat()

    current_date = date.fromisoformat(start_date)
    last_date = date.fromisoformat(end_date)

    success_runs = []
    failed_runs = []

    while current_date <= last_date:
        snapshot_date = current_date.isoformat()

        print(f"\n=== {snapshot_date} ===")

        try:
            # 1. Download
            download_result = download_epss(snapshot_date)

            # 2. Validate
            validation_result = validate_epss(
                download_result["content"]
            )

            # 요청 날짜와 실제 score_date 일치 확인
            if validation_result["snapshot_date"] != snapshot_date:
                raise ValueError(
                    f"snapshot_date mismatch: "
                    f"requested={snapshot_date}, "
                    f"actual={validation_result['snapshot_date']}"
                )

            # 3. Metadata
            metadata = build_metadata(
                download_result,
                validation_result
            )

            # 4. Fabric Bronze Save
            save_result = save_bronze_fabric(
                download_result["content"],
                metadata,
                bronze_dir=bronze_dir
            )

            success_runs.append({
                "snapshot_date": snapshot_date,
                "record_count": validation_result["record_count"],
                "batch_id": metadata["batch_id"],
                "checksum": metadata["checksum"],
                "raw_path": save_result["raw_path"],
                "meta_path": save_result["meta_path"]
            })

            print(
                f"[SUCCESS] "
                f"date={snapshot_date}, "
                f"rows={validation_result['record_count']}"
            )

        except Exception as e:
            failed_runs.append({
                "snapshot_date": snapshot_date,
                "error": str(e)
            })

            print(
                f"[FAILED] "
                f"date={snapshot_date}, "
                f"error={e}"
            )

        current_date += timedelta(days=1)

    return {
        "success_runs": success_runs,
        "failed_runs": failed_runs
    }

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 11, Finished, Available, Finished, False)

In [10]:
# 최초 1회만 실행 후 코드셀 실행하지말것!! 
"""
backfill_result = backfill_epss(
    start_date="2025-03-21",
    end_date="2026-08-19"
)
"""

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 12, Finished, Available, Finished, False)

'\nbackfill_result = backfill_epss(\n    start_date="2025-03-21",\n    end_date="2026-08-19"\n)\n'

## [3] 매일 1회 실행 셀

In [11]:
# ====================================
# run_daily_epss_ingestion()
# ====================================
def run_daily_epss_ingestion():
    download_result = download_epss()

    validation_result = validate_epss(
        download_result["content"]
    )

    metadata = build_metadata(
        download_result,
        validation_result
    )

    save_result = save_bronze_fabric(
        download_result["content"],
        metadata,
        bronze_dir="Files/bronze/01_epss"
    )

    print("EPSS ingestion completed")
    print("batch_id:", metadata["batch_id"])
    print("snapshot_date:", metadata["snapshot_date"])
    print("record_count:", metadata["record_count"])
    print("raw_path:", save_result["raw_path"])

    return {
        "metadata": metadata,
        "save_result": save_result
    }

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 13, Finished, Available, Finished, False)

In [12]:
# ====================================
# entry point
# ====================================

result = run_daily_epss_ingestion()

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 14, Finished, Available, Finished, False)

attempt=1, status=200
EPSS ingestion completed
batch_id: epss_20260825T013448Z
snapshot_date: 2026-08-24
record_count: 363908
raw_path: Files/bronze/01_epss/epss_2026-08-24_20260825T013448Z.csv.gz


In [13]:
# ================================================
# EPSS Backfill: 2026-08-20 ~ 오늘
# ================================================

start_date = date(2026, 8, 20)
end_date = date(2026, 8, 23)

current_date = start_date

while current_date <= end_date:
    target_date = current_date.isoformat()

    print(f"\n===== EPSS Backfill: {target_date} =====")

    try:
        # 1. 해당 날짜 EPSS 다운로드
        download_result = download_epss(
            snapshot_date=target_date
        )

        # 2. Raw 검증
        validation_result = validate_epss(
            download_result["content"]
        )

        # 3. Metadata 생성
        metadata = build_metadata(
            download_result,
            validation_result
        )

        # 4. Bronze 저장
        save_result = save_bronze_fabric(
            download_result["content"],
            metadata,
            bronze_dir="Files/bronze/01_epss"
        )

        print("SUCCESS")
        print("requested_date :", target_date)
        print("snapshot_date  :", metadata["snapshot_date"])
        print("record_count   :", metadata["record_count"])
        print("raw_path       :", save_result["raw_path"])

    except Exception as e:
        print("FAILED")
        print("requested_date :", target_date)
        print("error          :", str(e))

    current_date += timedelta(days=1)

StatementMeta(, 3f12901d-5634-40e7-b0d4-e6d00a1caf0d, 15, Finished, Available, Finished, False)


===== EPSS Backfill: 2026-08-20 =====
attempt=1, status=200
SUCCESS
requested_date : 2026-08-20
snapshot_date  : 2026-08-20
record_count   : 362881
raw_path       : Files/bronze/01_epss/epss_2026-08-20_20260825T013816Z.csv.gz

===== EPSS Backfill: 2026-08-21 =====
attempt=1, status=200
SUCCESS
requested_date : 2026-08-21
snapshot_date  : 2026-08-21
record_count   : 363376
raw_path       : Files/bronze/01_epss/epss_2026-08-21_20260825T013818Z.csv.gz

===== EPSS Backfill: 2026-08-22 =====
attempt=1, status=200
SUCCESS
requested_date : 2026-08-22
snapshot_date  : 2026-08-22
record_count   : 363588
raw_path       : Files/bronze/01_epss/epss_2026-08-22_20260825T013820Z.csv.gz

===== EPSS Backfill: 2026-08-23 =====
attempt=1, status=200
SUCCESS
requested_date : 2026-08-23
snapshot_date  : 2026-08-23
record_count   : 363829
raw_path       : Files/bronze/01_epss/epss_2026-08-23_20260825T013821Z.csv.gz
